In [13]:
import os
from openimages.download import download_dataset

In [14]:
data_dir = 'data'
number_of_samples = 400
classes = ['Missile', 'Balloon', 'Castle']

In [15]:
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

In [ ]:
if not (os.path.exists(os.path.join(data_dir, classes[0].lower())) and
        os.path.exists(os.path.join(data_dir, classes[1].lower())) and
        os.path.exists(os.path.join(data_dir, classes[2].lower()))):
    download_dataset(data_dir, classes, limit=number_of_samples)

In [ ]:
import torch
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import numpy as np
from skimage import io
from skimage.transform import resize
from skimage.color import gray2rgb
import glob

In [ ]:
def read_img(file_name):
    img = io.imread(file_name)
    if img.ndim == 2:
        img = gray2rgb(img)
    img = resize(img, (224, 224))
    img = torch.tensor(img)
    img = img.permute(2, 0, 1)
    return img.float()

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, images_dir):
        self.images_dir = images_dir
        self.transforms = transforms

        self.class1_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[0].lower()))
        self.class2_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[1].lower()))
        self.class3_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[2].lower()))
        self.class1_len = len(self.class1_files)
        self.class2_len = len(self.class2_files)
        self.class3_len = len(self.class3_files)

        self.files = self.class1_files + self.class2_files + self.class3_files

        self.labels = np.zeros(len(self.files))
        self.labels[self.class1_len:self.class1_len + self.class2_len] = 1
        self.labels[self.class1_len + self.class2_len:] = 2

        self.order = [x for x in np.random.permutation(len(self.labels))]
        self.files = [self.files[x] for x in self.order]
        self.labels = [self.labels[x] for x in self.order]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        file = self.files[i]
        im = read_img(file)

        img = im.clone().detach()

        y = self.labels[i]
        return img, y

In [ ]:
dataset = CustomDataset(data_dir)
train_dataset = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4, prefetch_factor=2)
iterator = iter(train_dataset)

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
model.eval()
pass

In [ ]:
class_indices = []
for cls in classes:
    idx = weights.meta['categories'].index(cls.lower())
    class_indices.append(idx)

In [ ]:
def prepare_batch(images):
    batch = []
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    for img in images:
        batch.append((img - mean) / std)

    return torch.cat(batch, dim=0)

In [ ]:
tp = [0, 0, 0]
fp = [0, 0, 0]
tn = [0, 0, 0]
fn = [0, 0, 0]

all_scores = []
all_labels = []

with torch.no_grad():
    for images, labels in iterator:
        batch = prepare_batch(images)
        predictions = model(batch).softmax(1)

        all_scores.append(predictions[:, class_indices])
        all_labels.append(labels)

all_scores = torch.cat(all_scores).numpy()
all_labels = torch.cat(all_labels).numpy()

In [ ]:
tp = np.zeros((100, 3))
fp = np.zeros((100, 3))
tn = np.zeros((100, 3))
fn = np.zeros((100, 3))

thresholds = np.arange(start=0.01, stop=1.01, step=0.01)

for i, threshold in enumerate(thresholds):
    for c in range(3):
        predicted_positive = all_scores[:, c] >= threshold
        actual_positive = all_labels == c

        tp[i, c] = (predicted_positive & actual_positive).sum()
        fp[i, c] = (predicted_positive & ~actual_positive).sum()
        tn[i, c] = (~predicted_positive & ~actual_positive).sum()
        fn[i, c] = (~predicted_positive & actual_positive).sum()

accuracy = (tp + tn) / (tp + fp + tn + fn)
precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) != 0)
recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) != 0)
f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) != 0)

print()
print('Classes: ', classes)
print('Mean accuracy: ', accuracy.mean(1) * 100)
print('Mean precision: ', precision.mean(1) * 100)
print('Mean recall: ', recall.mean(1) * 100)
print('Mean F1: ', f1.mean(1) * 100)


In [ ]:
import matplotlib.pyplot as plt

mean_f1 = f1.mean(axis=1) * 100

plt.plot(thresholds, mean_f1)
plt.xlabel('Threshold')
plt.ylabel('Mean F1 (%)')
plt.title('Mean F1 vs Threshold')
plt.show()

best_idx = mean_f1.argmax()
print(f'Best threshold: {thresholds[best_idx]:.2f}, Mean F1: {mean_f1[best_idx]:.2f}%')

In [ ]:
plt.title('Metrics vs Threshold')

plt.plot(thresholds, accuracy.mean(axis=1) * 100, label='Accuracy')
plt.plot(thresholds, precision.mean(axis=1) * 100, label='Precision')
plt.plot(thresholds, recall.mean(axis=1) * 100, label='Recall')
plt.plot(thresholds, f1.mean(axis=1) * 100, label='F1')

plt.xlabel('Threshold')
plt.ylabel('%')

plt.legend()
plt.show()